# Invoice-to-JSON Extractor
## Agentic Document Intelligence Pipeline

**Architecture:**
1. **Vision**: Ollama llama3.2-vision (local, free)
2. **Validation**: Pydantic schema enforcement
3. **Agentic Loop**: Math inconsistency detection
4. **Error Explain**: Groq LLM explanations

In [ ]:
import os, ollama, json, re, time, base64
from groq import Groq
from pydantic import BaseModel, Field, field_validator, ValidationError
from typing import Optional, List
from datetime import date, datetime
import dotenv
dotenv.load_dotenv()

In [12]:
# Initialize Groq
GROQ_API_KEY = os.getenv('GROQ_API_KEY')
if not GROQ_API_KEY:
    raise ValueError("❌ GROQ_API_KEY environment variable not set")
groq_client = Groq(api_key=GROQ_API_KEY)
print("✅ Groq connected")

✅ Groq connected


In [13]:
# Check Ollama vision model
models = ollama.list()
vision = [m['name'] for m in models.get('models', []) if 'vision' in m.get('name', '').lower()]
print(f"Vision models: {vision}")

Vision models: []


---
## Layer 1: Pydantic Schema

In [14]:
class LineItem(BaseModel):
    description: str = Field(..., min_length=1)
    quantity: float = Field(..., gt=0)
    unit_price: float = Field(..., ge=0)
    amount: float = Field(..., ge=0)
    
    @field_validator('amount')
    @classmethod
    def validate_amount(cls, v, info):
        if 'quantity' in info.data and 'unit_price' in info.data:
            expected = round(info.data['quantity'] * info.data['unit_price'], 2)
            if abs(v - expected) > 0.01:
                raise ValueError(f'Amount {v} != qty * price ({expected})')
        return v

In [15]:
class Invoice(BaseModel):
    # Required
    vendor_name: str = Field(..., min_length=1)
    customer_name: str = Field(..., min_length=1)
    invoice_number: str = Field(..., min_length=1)
    invoice_date: date
    subtotal: float = Field(..., ge=0)
    tax_rate: float = Field(..., ge=0, le=1)  # 0.08 not 8
    tax_amount: float = Field(..., ge=0)
    total_amount: float = Field(..., ge=0)
    line_items: List[LineItem] = Field(..., min_length=1)
    
    # Optional
    currency: str = Field(default="USD")
    due_date: Optional[date] = None
    shipping_amount: Optional[float] = Field(None, ge=0)
    discount_amount: Optional[float] = Field(None, ge=0)
    vendor_address: Optional[str] = None
    vendor_phone: Optional[str] = None
    vendor_email: Optional[str] = None
    vendor_tax_id: Optional[str] = None
    customer_address: Optional[str] = None
    customer_email: Optional[str] = None
    payment_terms: Optional[str] = None
    payment_method: Optional[str] = None
    bank_details: Optional[str] = None
    notes: Optional[str] = None

In [16]:
class ExtractionResult(BaseModel):
    success: bool
    invoice: Optional[Invoice] = None
    errors: List[str] = Field(default_factory=list)
    warnings: List[str] = Field(default_factory=list)
    flagged_fields: List[str] = Field(default_factory=list)
    processing_time_ms: Optional[float] = None

---
## Layer 2: Vision Extraction

In [17]:
EXTRACTION_PROMPT = """Extract invoice data. Return ONLY valid JSON.
Required: vendor_name, customer_name, invoice_number, invoice_date (YYYY-MM-DD), subtotal, tax_rate (DECIMAL 0.08 not 8), tax_amount, total_amount, currency, line_items (array of {description, quantity, unit_price, amount}).
Optional: due_date, shipping_amount, discount_amount, payment_terms, payment_method.
Return ONLY JSON object, no markdown."""

In [ ]:
def extract_from_image(image_path: str, max_retries: int = 3) -> dict:
    """Extract invoice data using Ollama vision model with retry logic"""
    # Read and encode image
    with open(image_path, 'rb') as f:
        image_data = base64.standard_b64encode(f.read()).decode('utf-8')

    last_error = None
    for attempt in range(max_retries):
        try:
            response = ollama.generate(
                model='llama3.2-vision',
                prompt=EXTRACTION_PROMPT,
                images=[image_data],
                options={'temperature': 0.0}
            )
            text = response['response'].strip()

            # Try multiple JSON extraction strategies
            # Strategy 1: Look for JSON block with ```json fences
            json_match = re.search(r'```json\s*(\{[\s\S]*?\})\s*```', text)
            if json_match:
                try:
                    return json.loads(json_match.group(1))
                except json.JSONDecodeError:
                    pass

            # Strategy 2: Look for any {...} block
            json_match = re.search(r'\{[\s\S]*\}', text)
            if json_match:
                try:
                    return json.loads(json_match.group())
                except json.JSONDecodeError as e:
                    last_error = f"Invalid JSON parsed: {e}"
                    continue

            # Strategy 3: Strip markdown code fences
            cleaned = re.sub(r'^```\s*', '', text.strip()).strip()
            if cleaned.startswith('{'):
                try:
                    return json.loads(cleaned)
                except json.JSONDecodeError:
                    pass

            last_error = f"No valid JSON found in response: {text[:200]}"
        except Exception as e:
            last_error = f"Extraction error: {str(e)}"

    raise ValueError(last_error or f"Failed after {max_retries} retries")

In [ ]:
def parse_date(value):
    """Parse date from various formats"""
    if value is None:
        return None
    if isinstance(value, date):
        return value

    if isinstance(value, str):
        value = value.strip()

        # Try ISO format first
        try:
            return date.fromisoformat(value)
        except (ValueError, TypeError):
            pass

        # Try common formats
        formats = [
            '%B %d, %Y',      # January 15, 2024
            '%d %B %Y',       # 15 January 2024
            '%B %d %Y',       # Jan 15 2024
            '%m/%d/%Y',       # 01/15/2024
            '%d/%m/%Y',       # 15/01/2024
            '%Y-%m-%d',       # 2024-01-15
            '%m-%d-%Y',       # 01-15-2024
            '%Y/%m/%d',       # 2024/01/15
        ]
        for fmt in formats:
            try:
                return datetime.strptime(value, fmt).date()
            except ValueError:
                continue

    return None

def clean_extracted_data(data: dict) -> dict:
    """Clean and normalize extracted data"""
    # Fix tax_rate > 1 (model returns 8 instead of 0.08)
    if 'tax_rate' in data and isinstance(data['tax_rate'], (int, float)) and data['tax_rate'] > 1:
        data['tax_rate'] = data['tax_rate'] / 100

    # Parse date strings to date objects
    for date_field in ['invoice_date', 'due_date']:
        if date_field in data and data[date_field]:
            parsed = parse_date(data[date_field])
            if parsed:
                data[date_field] = parsed
            # If still a string, let Pydantic handle it

    # Fix string numbers in line items
    for item in data.get('line_items', []):
        for field in ['quantity', 'unit_price', 'amount']:
            if field in item and isinstance(item[field], str):
                try:
                    item[field] = float(item[field].replace('$', '').replace(',', ''))
                except ValueError:
                    pass  # Keep original if conversion fails

    # Fix subtotal, tax_amount, total_amount if strings
    for field in ['subtotal', 'tax_amount', 'total_amount', 'shipping_amount', 'discount_amount']:
        if field in data and isinstance(data[field], str):
            try:
                data[field] = float(data[field].replace('$', '').replace(',', ''))
            except ValueError:
                pass

    # Fix negative discount amounts (treat as 0)
    if 'discount_amount' in data and isinstance(data['discount_amount'], (int, float)) and data['discount_amount'] < 0:
        data['discount_amount'] = 0.0

    return data

---
## Layer 3: Validation & Math Checks

In [20]:
def validate_and_parse(raw_data: dict) -> ExtractionResult:
    cleaned = clean_extracted_data(raw_data)
    try:
        invoice = Invoice(**cleaned)
        return ExtractionResult(success=True, invoice=invoice)
    except ValidationError as e:
        errors, flagged = [], []
        for err in e.errors():
            field = '.'.join(str(l) for l in err['loc'])
            errors.append(f"'{field}': {err['msg']}")
            flagged.append(field)
        return ExtractionResult(success=False, errors=errors, flagged_fields=flagged)
    except Exception as e:
        return ExtractionResult(success=False, errors=[str(e)])

In [21]:
def check_math_consistency(invoice: Invoice) -> List[str]:
    """Agentic check: detect math inconsistencies"""
    warnings = []
    
    # Check line items vs subtotal
    calc_sub = sum(item.amount for item in invoice.line_items)
    if abs(calc_sub - invoice.subtotal) > 0.02:
        warnings.append(f"SUBTOTAL: declared={invoice.subtotal}, calc={calc_sub}")
    
    # Check tax calculation
    calc_tax = invoice.subtotal * invoice.tax_rate
    if abs(calc_tax - invoice.tax_amount) > 0.02:
        warnings.append(f"TAX: declared={invoice.tax_amount}, calc={calc_tax:.2f}")
    
    # Check total
    calc_total = invoice.subtotal + invoice.tax_amount
    if invoice.shipping_amount: calc_total += invoice.shipping_amount
    if invoice.discount_amount: calc_total -= invoice.discount_amount
    calc_total = round(calc_total, 2)
    if abs(calc_total - invoice.total_amount) > 0.02:
        warnings.append(f"TOTAL: declared={invoice.total_amount}, calc={calc_total}")
    
    return warnings

In [22]:
def explain_errors(errors: List[str]) -> str:
    prompt = "Invoice errors:\n" + "\n".join([f"- {e}" for e in errors]) + "\n\nExplain simply."
    resp = groq_client.chat.completions.create(
        model="llama-3.3-70b-versatile",
        messages=[{"role": "user", "content": prompt}],
        max_tokens=400
    )
    return resp.choices[0].message.content

---
## Layer 4: Full Pipeline

In [23]:
def process_invoice(image_path: str) -> ExtractionResult:
    """Full pipeline: Extract -> Validate -> Check Math -> Explain"""
    start = time.time()
    
    # Validate file exists
    if not os.path.exists(image_path):
        return ExtractionResult(
            success=False, 
            errors=[f"Image file not found: {image_path}"],
            processing_time_ms=round((time.time() - start) * 1000, 2)
        )
    
    try:
        # Step 1: Extract
        raw = extract_from_image(image_path)
    except Exception as e:
        return ExtractionResult(
            success=False,
            errors=[f"Extraction failed: {str(e)}"],
            processing_time_ms=round((time.time() - start) * 1000, 2)
        )
    
    # Step 2: Validate
    result = validate_and_parse(raw)
    result.processing_time_ms = round((time.time() - start) * 1000, 2)
    
    # Step 3: Math check
    if result.success:
        result.warnings = check_math_consistency(result.invoice)
        result.flagged_fields = result.warnings.copy()
    
    # Step 4: Explain errors
    if not result.success and result.errors:
        try:
            result.errors.append("\n" + explain_errors(result.errors))
        except Exception as e:
            result.errors.append(f"\n(Error explanation failed: {str(e)})")
    
    return result

---
## Testing

In [ ]:
# Create output directory (use relative paths)
OUTPUT_DIR = 'output'
DATASET_DIR = 'dataset'
os.makedirs(OUTPUT_DIR, exist_ok=True)

print("=" * 70)
print("INVOICE EXTRACTION PIPELINE - BATCH PROCESSING")
print("=" * 70)

# Process all invoices in dataset
dataset_path = DATASET_DIR
invoice_files = sorted([f for f in os.listdir(dataset_path) if f.lower().endswith(('.jpg', '.jpeg', '.png'))])

results = []
for invoice_file in invoice_files:
    image_path = os.path.join(dataset_path, invoice_file)
    print(f"\n📄 Processing: {invoice_file}...")
    
    result = process_invoice(image_path)
    results.append({'file': invoice_file, 'result': result})
    
    print(f"   Success: {result.success} | Time: {result.processing_time_ms}ms")
    
    if result.success:
        inv = result.invoice
        print(f"   ✅ {inv.vendor_name} -> {inv.customer_name}")
        print(f"      Invoice: {inv.invoice_number} | Date: {inv.invoice_date}")
        print(f"      Total: {inv.currency} {inv.total_amount}")
        
        # Save to JSON (use model_dump instead of model_dump_json round-trip)
        output_file = os.path.join(OUTPUT_DIR, f"{invoice_file.split('.')[0]}.json")
        with open(output_file, 'w') as f:
            json.dump(inv.model_dump(mode='json'), f, indent=2, default=str)
        print(f"      💾 Saved: {output_file}")
        
        if result.warnings:
            print(f"      ⚠️ Warnings: {len(result.warnings)}")
            for w in result.warnings:
                print(f"         - {w}")
    else:
        print(f"   ❌ Errors:")
        for e in result.errors[:2]:
            print(f"      {e[:100]}")

print("\n" + "=" * 70)
print(f"✅ BATCH COMPLETE: {sum(1 for r in results if r['result'].success)}/{len(results)} successful")
print(f"📁 Output directory: {OUTPUT_DIR}")
print("=" * 70)